[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/kalman/kalman.ipynb)

# Tracking a Pose with a Kalman Filter

This example tracks a vehicle's pose in planar space with a Kalman filter. As the vehicle executes forward motion and turning commands, wheel slip and heading noise cause dead-reckoned odometry to drift steadily away from the true trajectory, while occasional noisy readings observe the full position and heading together. The filter advances its pose estimate with each command, tracks how uncertain that estimate has grown, and pulls the estimate toward each incoming pose reading in proportion to the two uncertainties.

The pose is modelled as a motor, and an error in it as a small twist perturbation on its right. A linear readout of a twist is an antibivector line paired with it through `line & twist`, where the term ***duality map*** here means a linear map between a space and its dual readout space; pose uncertainty is a covariance duality map `Twist <- Line` sending each readout line to its expected twist so `line & sigma(line)` gives that readout's variance. Propagating through a commanded step pulls the readout line forward and pushes the twist back via `step << sigma(step >> Line)` without deriving a coordinate Jacobian, the Kalman gain at a reading is a ratio of two covariance maps `sigma((sigma + measurement_noise).solve(Twist))`, and pushing the covariance through `Twist.commutator(positions)` yields the implicit confidence ellipse of the vehicle's centre.

In the matrix notation of the extended Kalman filter on $\mathrm{SE}(2)$, the pose state $(x, y, \theta)$ carries a $3 \times 3$ error covariance $P$ in the body Lie algebra $\mathfrak{se}(2)$, propagated as $F P F^{\mathsf T} + Q$ using the $3 \times 3$ adjoint matrix $F = \operatorname{Ad}_{\Delta g^{-1}}$ assembled from $\cos\Delta\theta$, $\sin\Delta\theta$, and cross-coupled translation offsets, updated with the matrix gain $K = P (P + R)^{-1}$ on the logarithmic coordinates $\log(\hat g^{-1} z)^\vee$, and projected to world position via a $2 \times 3$ heading-dependent Jacobian $J(\hat\theta)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from examples.estimation.kalman import render
from examples.util.animation import show_states
from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
# A linear readout of a twist is an antibivector line; a covariance duality map sends it to a twist.
Line = ga.gatype.antibivector()                     # Antibivector-
# A pose perturbation on the right carrying the estimated motor toward the true motor.
Twist = ga.gatype.bivector()                        # Bivector+
Motor = ga.gatype.rotor()                           # Rotor+
Covariance = ga.gatype((Twist, Line))               # Twist+ <- Line-
Quadric = ga.gatype((Plane, Point))                 # Plane+ <- Point-
# Unit weight point at the origin.
origin = mv.w.dual()                                # [] Point-

## Predicting and updating on the motor manifold

Each motion step multiplies the pose estimate on the right by the commanded motor and transports the body-frame covariance map across that step via `step << sigma(step >> Line)`, adding the step's motion noise. When a pose reading arrives, the relative motor from the estimate to the reading has logarithmic twist `innovation`; the ratio of covariance maps `sigma((sigma + measurement_noise).solve(Twist))` maps that twist to the correction twist applied on the right and reduces the covariance by the information gained. Both motion and measurement noises combine isotropic body-frame translation dyads with rotation dyads about the body origin.

In [ ]:
def body_covariance(translation_std: float, rotation_std: float) -> Covariance:
    """Translation variance `translation_std**2` along both body axes and rotation variance `rotation_std**2`
    about the body origin form an isotropic body-frame covariance (`[] Twist+ <- Line-`)."""
    translation = mv.yw * (mv.yw & Line) + mv.wx * (mv.wx & Line)               # [] Twist+ <- Line-
    rotation = mv.xy * (mv.xy & Line)                                           # [] Twist+ <- Line-
    return translation * translation_std**2 + rotation * rotation_std**2


def filter_update(
    estimate: Motor, sigma: Covariance, reading: Motor, measurement_noise: Covariance,
) -> tuple[Motor, Covariance]:
    """A noisy pose `reading` (`[] Motor+`) with right-perturbation covariance `measurement_noise`
    (`[] Twist+ <- Line-`) updates `estimate` and shrinks `sigma` (`[] Motor+`, `[] Twist+ <- Line-`)."""
    # Relative twist from the current estimate to the reading, weighted by the ratio of covariance maps.
    innovation = (estimate.inverse() * reading).log() * 2                       # [] Twist+
    gain = sigma((sigma + measurement_noise).solve(Twist))                      # [] Twist+ <- Twist+
    correction = (gain(innovation) * 0.5).exp()                                 # [] Motor+
    return estimate * correction, sigma - gain(sigma)

## Position confidence quadric and meandering drive

Although the filter maintains uncertainty on three-dimensional twists in the body frame, spatial position uncertainty follows directly from how a body twist displaces the vehicle's centre in world space. Carrying body twists to world space by `estimate >> Twist` and taking `Twist.commutator(positions)` gives the linear motion map from twists to point displacements, whose `.adjugate()` pulls displacement-measuring planes back to twist readout lines; inverting the resulting second-moment dual quadric yields the implicit confidence ellipse. Sampling synthetic noise twists uses readout lines orthonormal in the covariance's own bilinear form `Line & noise`.

In [ ]:
# Timestep in seconds, reading count, commanded steps between readings, and random seed.
dt, n_readings, n_steps_per_reading, seed = 0.1, 12, 25, 2
# Sinusoidal commanded turn-rate amplitude and angular frequency (rad/s).
turn_amplitude, turn_frequency = 0.9, 0.2
# Translation and rotation standard deviations: per square-root second for motion, and per pose reading.
motion_std = np.array([0.05, 0.12])
measurement_std = np.array([0.3, 0.1])
sigmas = 2.0
frame_duration_ms = 40

In [ ]:
def position_ellipsoid(estimate: Motor, sigma: Covariance, positions: Point, sigmas: float) -> Quadric:
    """Pushing `sigma` (`[...] Twist+ <- Line-`) through the body-twist motion of `positions` (`[...] Point-`)
    gives the implicit confidence quadrics (`[...] Plane+ <- Point-`) at `sigmas` standard deviations."""
    # Body-frame twists carried to world space move the vehicle centre by their commutator with its position.
    shift = Twist.commutator(positions)(estimate >> Twist)                      # [...] Point- <- Twist+
    # Pulling each displacement-measuring plane back through the motion gives the corresponding twist readout line.
    readout = shift.adjugate()                                                  # [...] Line- <- Plane+
    # Adding the pushed-forward position covariance to the position's own dyad forms its second-moment dual quadric.
    moment = positions * (Plane & positions) + shift(sigma(readout))            # [...] Point- <- Plane+
    return moment.solve(Point) - (1 + sigmas**2) * mv.w * (mv.w & Point)        # [...] Plane+ <- Point-


def twist_samples(noise: Covariance, normals: np.ndarray) -> Twist:
    """Standard normal draws `normals` (`[..., 3]`) along readout lines orthonormal in `noise` (`[] Twist+ <- Line-`)
    produce twist perturbations (`[...] Twist+`) with covariance `noise`."""
    readouts = Line & noise                                                     # [] Scalar+ <- (Line-, Line-)
    # Generalized eigenvectors of the readout form with itself are readout lines orthonormal under that form.
    _, lines = readouts.eigh(readouts)                                          # [3] Line-
    return (noise(lines) * normals).sum(axis=-1)                                # [...] Twist+

In [ ]:
def drive_states(
    initial: Motor,
    increments: Twist,
    motion_noise: Covariance,
    measurement_noise: Covariance,
    rng: np.random.Generator,
) -> Iterator[tuple[Motor, Motor, Motor, Covariance, Motor]]:
    """Yields `(truth, dead, estimate, sigma, reading)` after each commanded step in `increments`
    (`[n_readings, n_steps_per_reading] Twist+`) and after each pose reading."""
    truth = dead = estimate = reading = initial
    sigma = body_covariance(0.0, 0.0)                                           # [] Twist+ <- Line-
    for segment in increments:
        for increment in segment:
            step = (increment * 0.5).exp()                                      # [] Motor+
            noisy_step = ((increment + twist_samples(motion_noise, rng.normal(size=3))) * 0.5).exp()   # [] Motor+
            truth = truth * noisy_step
            dead = dead * step
            # Pull the readout line through the step and push the expected twist back.
            estimate, sigma = estimate * step, (step << sigma(step >> Line)) + motion_noise
            yield truth, dead, estimate, sigma, reading
        measurement_error = (twist_samples(measurement_noise, rng.normal(size=3)) * 0.5).exp()   # [] Motor+
        reading = truth * measurement_error
        estimate, sigma = filter_update(estimate, sigma, reading, measurement_noise)
        yield truth, dead, estimate, sigma, reading


times = np.arange(n_readings * n_steps_per_reading) * dt
turn = turn_amplitude * np.sin(turn_frequency * times)
increments = ((mv.xy * turn - mv.wx) * dt).reshape(n_readings, n_steps_per_reading)   # [n_readings, n_steps_per_reading] Twist+
motion_noise = body_covariance(*(motion_std * np.sqrt(dt)))                     # [] Twist+ <- Line-
measurement_noise = body_covariance(*measurement_std)                           # [] Twist+ <- Line-
ellipsoid_of = partial(position_ellipsoid, sigmas=sigmas)
n_frames = n_readings * (n_steps_per_reading + 1)
recorded = show_states(
    drive_states(mv.rotor(), increments, motion_noise, measurement_noise, np.random.default_rng(seed)),
    render.tracking_animation,
    increments,
    ellipsoid_of,
    name="kalman",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

The black curve traces the true path, the dashed red curve traces dead reckoning, and the blue curve traces the Kalman filter's estimate, each tipped by its triangular vehicle outline; green dots mark the noisy pose readings received so far, and the shaded blue ellipse shows the filter's two-standard-deviation position uncertainty swelling as heading uncertainty compounds into cross-track spread between readings and contracting sharply at each update.

In [ ]:
truth, dead, estimates, covariances, readings = (stack(part) for part in zip(*recorded))   # [n_frames] Motor+, Motor+, Motor+, Twist+ <- Line-, Motor+
true_positions = truth >> origin                                                # [n_frames] Point-
positions = estimates >> origin                                                 # [n_frames] Point-
# For unit-weight points, the norm of their joining line equals their Euclidean separation.
dead_error = (true_positions & (dead >> origin)).norm()                         # [n_frames] Scalar+
filtered_error = (true_positions & positions).norm()                            # [n_frames] Scalar+
print("motion noise per step, a duality map from lines to twists:\n", motion_noise.coefficient_table())
print("covariance after the last reading:\n", covariances[-1].coefficient_table())
print("mean position error, dead reckoning:", dead_error.mean(axis=0).to_array())
print("mean position error, filtered:      ", filtered_error.mean(axis=0).to_array())

In [ ]:
# checks
# Sparse noisy readings keep the filtered path far closer to the truth than dead reckoning,
# every covariance stays symmetric under `.adjugate()`, and each reading update reduces uncertainty.
assert bool(filtered_error.mean(axis=0) < 0.25 * dead_error.mean(axis=0))
frame = mv.bivector(np.eye(3)).dual()                                           # [3] Line-
skew = covariances.adjugate() - covariances                                     # [n_frames] Twist+ <- Line-
symmetry_residual = (frame[:, None] & skew[..., None, None](frame[None, :])).abs().max()   # [] Scalar+
assert bool(symmetry_residual < 1e-12)
update_frames = np.arange(1, n_readings + 1) * (n_steps_per_reading + 1) - 1
before_trace = Twist.dual()(covariances[update_frames - 1]).trace()             # [n_readings] Scalar+
after_trace = Twist.dual()(covariances[update_frames]).trace()                  # [n_readings] Scalar+
assert bool((after_trace < before_trace).all())